# Ejecución local

Hasta ahora, todos los circuitos se han ejecutado con `infrastructure="local"`. Este notebook explica qué hace ese backend, cómo repartir shots entre varias ejecuciones con `n_qpus`, y cómo fijar la aleatoriedad con `seed`. La sección siguiente cubre las infraestructuras que necesitan un clúster: CUNQA y QMIO.

## Qué hay detrás de "local"

`infrastructure="local"` ejecuta por defecto el circuito con el simulador Aer de Qiskit, en el propio ordenador. `result.backend` lo confirma:

In [ ]:
import polypus

bell = polypus.Circuit(2).h(0).cx(0, 1).measure_all()
result = polypus.run_quantum_circuit(bell, shots=1000, infrastructure="local")
print(result.backend)

## El backend nativo

Además de `"aer"`, Polypus tiene un motor de simulación propio en Rust, seleccionable con el argumento `backend`:

In [ ]:
result_nativo = polypus.run_quantum_circuit(
    bell, shots=1000, infrastructure="local", backend="polypus"
)
print(result_nativo.backend, result_nativo.counts)

El resultado es el mismo estado de Bell, solo cambia qué motor lo calcula. El backend nativo también admite `fusion`, que combina puertas consecutivas antes de simular para ir más rápido: se activa por defecto, y `fusion=False` fuerza una ejecución puerta a puerta. Pasar `fusion=True` en `"aer"`, que no sabe fusionar, da un error en vez de ignorarlo en silencio:

In [ ]:
try:
    polypus.run_quantum_circuit(
        bell, shots=1000, infrastructure="local", backend="aer", fusion=True
    )
except ValueError as e:
    print(e)

Las diferencias de rendimiento entre ambos backends están documentadas, con benchmarks reproducibles, en el README principal.

## Repartir shots con `n_qpus`

`n_qpus` reparte la ejecución de los shots entre varias instancias en paralelo, para reducir el tiempo de reloj. Con `n_qpus=1`, el valor por defecto, el resultado viene envuelto en una lista de un único elemento, como se vio en la sección 1:

In [ ]:
r1 = polypus.run_quantum_circuit(bell, shots=1000, infrastructure="local", n_qpus=1)
print(r1.counts)

Con `n_qpus` mayor que 1, Polypus fusiona los resultados de todas las instancias en un único diccionario, sin lista:

In [ ]:
r4 = polypus.run_quantum_circuit(bell, shots=1000, infrastructure="local", n_qpus=4)
print(r4.counts)

Este cambio de forma es un motivo real de error: código que da por hecho `result.counts[0]` deja de funcionar en cuanto `n_qpus` pasa de 1 a un valor mayor, y salta un `KeyError`, porque un diccionario de bitstrings no tiene ninguna clave `0`:

In [ ]:
try:
    r4.counts[0]
except KeyError as e:
    print("KeyError:", e)

Por eso conviene comprobar el tipo antes de acceder, o simplemente tratar `result.counts` como lo que es en cada caso: una lista con un elemento, o un diccionario ya fusionado.

## Reproducibilidad con `seed`

Por defecto, cada ejecución usa una semilla aleatoria distinta, tomada del sistema operativo. Pasar `seed` fija esa aleatoriedad, y el resultado es idéntico entre ejecuciones:

In [ ]:
a = polypus.run_quantum_circuit(bell, shots=1000, infrastructure="local", seed=42)
b = polypus.run_quantum_circuit(bell, shots=1000, infrastructure="local", seed=42)
print(a.counts == b.counts)

Sin pasar `seed`, `result.seed` informa de la semilla que se usó, para poder reproducir esa misma ejecución más tarde:

In [ ]:
sin_semilla = polypus.run_quantum_circuit(bell, shots=1000, infrastructure="local")
print(sin_semilla.seed)

## Otros campos del resultado

Además de `counts`, `seed` y `backend`, `result.id` guarda un identificador único de la ejecución, y `result.infrastructure` repite el backend usado. Son útiles para registrar y encontrar una ejecución concreta más adelante.

## Resumen

Este notebook ha explicado qué hace `infrastructure="local"`, la elección entre el backend `"aer"` y el nativo `"polypus"` con `fusion`, cómo `n_qpus` cambia tanto el reparto de shots como la forma del resultado, con el cambio de lista a diccionario fusionado como fuente real de errores, y cómo `seed` da reproducibilidad.

La siguiente sección trata las infraestructuras que necesitan un clúster: CUNQA y QMIO.

## Siguiente paso

Continúa con: [`03b_escalar_cunqa_qmio.ipynb`](03b_escalar_cunqa_qmio.ipynb).